# Phase 0 — Folder Setup

**Notebook:** `notebooks/00_setup/01_folder_setup.ipynb`

First notebook in the thesis. Creates the canonical folder structure on Google Drive, drops in starter files (README, requirements, .gitignore, minimal `src/` modules, an example config), and verifies that `src/` is importable.

### When to run this

- **Once**, at the very start of the project.
- Safe to re-run — operations are idempotent (`mkdir(..., exist_ok=True)`, file writes only happen if the file doesn't already exist).

### What gets created

```
phd_thesis/
├── README.md
├── requirements.txt
├── .gitignore
├── configs/{data,experiments,models}/
├── data/{raw,interim,processed}/
├── src/{data,models,explainers,divergence,detectors,evaluation,visualization,utils}/
├── notebooks/00_setup … 99_paper_figures/
├── experiments/{logs,results,checkpoints}/
├── reports/{figures,tables,advisor_notes}/
├── tests/
├── papers/01_negative_result_workshop/
└── thesis/chapters/
```

### Design rules encoded by this layout

1. Notebooks orchestrate — they import from `src/`. Logic does **not** live in notebooks.
2. Data has three states: `raw` (sacred, never edited), `interim` (cleaned/joined), `processed` (analysis-ready).
3. Every experiment is described by a YAML config in `configs/experiments/`. Re-running an experiment = re-running its YAML.
4. All plots come from `src/visualization/plots.py`. No matplotlib calls inside notebooks unless throwaway exploration.
5. Tests in `tests/` should pass before changes to `src/` are committed.

## 1. Mount Drive and choose root

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import textwrap, sys

ROOT = Path('/content/drive/MyDrive/phd_thesis')
ROOT.mkdir(parents=True, exist_ok=True)
print(f'Thesis root: {ROOT}')

## 2. Create the directory tree

In [ ]:
DIRS = [
    'configs/data',
    'configs/experiments',
    'configs/models',
    'data/raw',
    'data/interim',
    'data/processed',
    'src/data',
    'src/models',
    'src/explainers',
    'src/divergence',
    'src/detectors',
    'src/evaluation',
    'src/visualization',
    'src/utils',
    'notebooks/00_setup',
    'notebooks/01_phase0_foundation',
    'notebooks/02_phase1_synthetic',
    'notebooks/03_phase1_cicids',
    'notebooks/04_phase2_multidataset',
    'notebooks/05_phase3_generalization',
    'notebooks/06_phase4_mechanism',
    'notebooks/99_paper_figures',
    'experiments/logs',
    'experiments/results',
    'experiments/checkpoints',
    'reports/figures',
    'reports/tables',
    'reports/advisor_notes',
    'tests',
    'papers/01_negative_result_workshop',
    'thesis/chapters',
]

for d in DIRS:
    (ROOT / d).mkdir(parents=True, exist_ok=True)

print(f'Created or verified {len(DIRS)} directories.')

## 3. Make `src/` a proper Python package

Empty `__init__.py` files in every subpackage so we can `from src.utils import seeding` from any notebook.

In [ ]:
INIT_PACKAGES = [
    'src', 'src/data', 'src/models', 'src/explainers',
    'src/divergence', 'src/detectors', 'src/evaluation',
    'src/visualization', 'src/utils',
]
for p in INIT_PACKAGES:
    (ROOT / p / '__init__.py').touch()

print(f'Touched {len(INIT_PACKAGES)} __init__.py files.')

## 4. Drop in starter files

Each `write()` call is a no-op if the file already exists — re-running won't overwrite your edits.

In [ ]:
def write(path_rel: str, content: str):
    p = ROOT / path_rel
    if not p.exists():
        p.write_text(textwrap.dedent(content).lstrip())
        print(f'  [new]  {path_rel}')
    else:
        print(f'  [skip] {path_rel} (already exists)')

write('README.md', """
    # PhD Thesis — Explanation-Space Dynamics under Concept Drift in IDS

    ## Layout
    - `src/` — Python modules (the actual science)
    - `notebooks/` — narrative orchestration, numbered by phase
    - `configs/` — YAML configs, one per experiment
    - `data/` — raw / interim / processed (raw is sacred, never edited)
    - `experiments/` — logs, results, checkpoints
    - `reports/` — figures, tables, advisor notes
    - `tests/` — pytest tests for src/
    - `papers/` — paper drafts
    - `thesis/` — final thesis document

    ## Rules
    1. Notebooks import from `src/`. Notebooks do not contain logic.
    2. Every experiment is a YAML config + a run script.
    3. Every random operation uses a seed from the config.
    4. Raw data is read-only. Always derive interim/processed from raw.
    5. Plots are generated by `src/visualization/plots.py`.
    6. Tests in `tests/` pass before committing changes to `src/`.
""")

write('requirements.txt', """
    numpy==1.26.4
    pandas==2.2.2
    scikit-learn==1.5.1
    scipy==1.13.1
    matplotlib==3.9.2
    seaborn==0.13.2
    shap==0.46.0
    lime==0.2.0.1
    river==0.21.2
    pyyaml==6.0.2
    pyarrow==17.0.0
    mlflow==2.16.0
    pytest==8.3.3
    tqdm==4.66.5
    xgboost==2.1.1
    kaggle==1.6.17
""")

write('.gitignore', """
    # Data — never commit data
    data/raw/*
    data/interim/*
    data/processed/*
    !data/raw/.gitkeep
    !data/interim/.gitkeep
    !data/processed/.gitkeep

    # Experiment outputs
    experiments/logs/*
    experiments/checkpoints/*
    !experiments/logs/.gitkeep
    !experiments/checkpoints/.gitkeep

    # Kaggle token — never commit credentials
    .kaggle/
    kaggle.json

    # Python
    __pycache__/
    *.pyc
    .ipynb_checkpoints/
    .pytest_cache/
    *.egg-info/
""")

# Keep empty dirs trackable in git
for d in ['data/raw', 'data/interim', 'data/processed',
         'experiments/logs', 'experiments/checkpoints']:
    (ROOT / d / '.gitkeep').touch()

## 5. Minimal `src/` modules

Just enough so `import` works. We'll grow `src/` properly in later notebooks.

In [ ]:
write('src/utils/seeding.py', """
    import random, numpy as np, os

    def set_global_seed(seed: int = 42) -> None:
        random.seed(seed)
        np.random.seed(seed)
        os.environ['PYTHONHASHSEED'] = str(seed)
        try:
            import torch
            torch.manual_seed(seed)
            torch.cuda.manual_seed_all(seed)
        except ImportError:
            pass
""")

write('src/utils/config.py', """
    from pathlib import Path
    import yaml

    def load_config(path):
        with open(path, 'r') as f:
            return yaml.safe_load(f)

    def save_config(cfg: dict, path) -> None:
        Path(path).parent.mkdir(parents=True, exist_ok=True)
        with open(path, 'w') as f:
            yaml.safe_dump(cfg, f, sort_keys=False)
""")

write('src/utils/io.py', """
    from pathlib import Path

    THESIS_ROOT = Path('/content/drive/MyDrive/phd_thesis')

    def path(*parts) -> Path:
        '''Build paths anchored to thesis root.'''
        return THESIS_ROOT.joinpath(*parts)
""")

write('configs/experiments/example.yaml', """
    name: example_run
    seed: 42
    dataset: cicids2017_improved
    drift_type: day_split
    model:
      type: random_forest
      n_estimators: 100
      max_depth: 10
    auxiliary:
      window: 2000
    explainer: shap_tree
    divergence: wasserstein
    output_dir: experiments/results/example_run
""")

write('tests/test_smoke.py', """
    def test_imports():
        from src.utils import seeding, config, io
        seeding.set_global_seed(0)
        assert io.THESIS_ROOT.exists()
""")

## 6. Verify the structure and import path

In [ ]:
# Add to sys.path so we can import src.* from this notebook
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Print the tree (depth 2)
def tree(p: Path, prefix: str = '', depth: int = 0, max_depth: int = 2):
    if depth > max_depth:
        return
    entries = sorted([e for e in p.iterdir() if not e.name.startswith('.')])
    for i, e in enumerate(entries):
        last = i == len(entries) - 1
        print(prefix + ('└── ' if last else '├── ') + e.name)
        if e.is_dir():
            tree(e, prefix + ('    ' if last else '│   '), depth + 1, max_depth)

print(f'{ROOT.name}/')
tree(ROOT)

# Sanity check imports
from src.utils import seeding, io, config
seeding.set_global_seed(42)
print(f'\nImport check passed. THESIS_ROOT = {io.THESIS_ROOT}')

example = config.load_config(ROOT / 'configs/experiments/example.yaml')
print(f'Example config loaded: {list(example.keys())}')

## 7. Next steps

1. Open `02_download_data.ipynb` and run it to download the four datasets.
2. After datasets are in place, write `src/data/loaders.py` (the unified loader interface for all datasets).
3. Write the first data-characterization notebook in `notebooks/01_phase0_foundation/`.

### Future sessions

Once this notebook has been run, every future Colab session should start with this preamble in cell 1:

```python
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.insert(0, '/content/drive/MyDrive/phd_thesis')

%pip install -q -r /content/drive/MyDrive/phd_thesis/requirements.txt
```

Then `from src.data.loaders import ...` and you're working.